# Prompt Engineering Lab

## Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODEL = "gemini-3.1-flash-lite"
THINKING = types.ThinkingConfig(thinking_level=types.ThinkingLevel.MINIMAL)


## Task 1: Summarization

### Iteration 1 — Baseline

In [ ]:
text_to_summarize = (
    "The city council voted 6-3 on Tuesday to approve a $42 million budget for repaving "
    "roughly 38 miles of residential streets over the next two years. Supporters said the "
    "plan targets neighborhoods that have not seen major roadwork in over a decade and "
    "will reduce vehicle repair costs and improve emergency-vehicle access. Opponents "
    "argued the funding should instead go toward public transit expansion, noting that "
    "only 12% of the city's residents drive to work daily."
)

response = client.models.generate_content(
    model=MODEL,
    contents="Summarize this: " + text_to_summarize,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=1.0,
        top_p=0.95,
    ),
)
print(response.text)


### Iteration 2 — Add context, constraints, lower temperature

In [ ]:
refined_prompt = (
    "You are writing for a local newspaper's 60-second briefing section. "
    "Summarize the following article in exactly 2 sentences, in a neutral tone. "
    "Do not add opinions or information not in the source text.\n\n"
    + text_to_summarize
)

response = client.models.generate_content(
    model=MODEL,
    contents=refined_prompt,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        system_instruction="You are an expert news editor.",
        temperature=0.2,
        top_p=0.8,
    ),
)
print(response.text)


## Task 2: Email Generation

### Iteration 1 — Baseline

In [ ]:
customer_email = (
    "Subject: Order #48213 arrived damaged\n\n"
    "Hi, I ordered a ceramic vase (Order #48213) last week and it arrived today with a "
    "large crack down one side. I paid for expedited shipping because this was meant to "
    "be a birthday gift for this weekend. I would like a replacement shipped overnight, "
    "or a full refund if that is not possible. Please let me know quickly.\n\n- Jordan"
)

response = client.models.generate_content(
    model=MODEL,
    contents="Write a reply to this email: " + customer_email,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=1.0,
        top_p=0.95,
    ),
)
print(response.text)


### Iteration 2 — Add role, policy context, constraints

In [ ]:
company_policy = (
    "For damaged items reported within 14 days, offer either (a) a free overnight "
    "replacement at no extra cost, or (b) a full refund including original shipping. "
    "Apologize once, be concise, and always restate both options clearly."
)

refined_prompt = f"""Using the policy below, write a reply to the customer email below.

Policy:
{company_policy}

Customer email:
{customer_email}

Constraints:
- Apologize exactly once.
- Clearly restate BOTH options as separate bullet points.
- Keep the email under 120 words.
- Sign off as "The Support Team".
"""

response = client.models.generate_content(
    model=MODEL,
    contents=refined_prompt,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        system_instruction="You are a customer support agent for an online home-goods store.",
        temperature=0.4,
        top_p=0.9,
    ),
)
print(response.text)


## Task 3: Data Analysis with Structured (JSON) Output

### Iteration 1 — Baseline (free text)

In [ ]:
sales_table = """
Region,Q1_Sales,Q2_Sales
North,120000,135000
South,98000,91000
East,150000,162000
West,87000,79000
"""

response = client.models.generate_content(
    model=MODEL,
    contents="Analyze this sales data: " + sales_table,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=1.0,
        top_p=0.95,
    ),
)
print(response.text)


### Iteration 2 — Structured JSON output with a schema

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "regions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "region": {"type": "string"},
                    "q1_sales": {"type": "number"},
                    "q2_sales": {"type": "number"},
                    "pct_change": {"type": "number"},
                },
                "required": ["region", "q1_sales", "q2_sales", "pct_change"],
            },
        },
        "best_performing_region": {"type": "string"},
        "worst_performing_region": {"type": "string"},
    },
    "required": ["regions", "best_performing_region", "worst_performing_region"],
}

prompt = (
    "Analyze the sales data below. pct_change = (Q2 - Q1) / Q1 * 100, rounded to 1 decimal.\n\n"
    + sales_table
)

response = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config=types.GenerateContentConfig(
        thinking_config=THINKING,
        temperature=0.1,
        response_mime_type="application/json",
        response_schema=schema,
    ),
)

parsed = json.loads(response.text)
print(json.dumps(parsed, indent=2))
